In [1]:
import pandas as pd
import numpy as np
# Load dataset
dataset = pd.read_csv("CKD.csv")

# Normalize column names
dataset.columns = dataset.columns.str.strip().str.lower()

# Clean & encode target
dataset['classification'] = dataset['classification'].astype(str).str.strip().str.lower()
dataset = dataset[dataset['classification'].isin(['yes', 'no'])]
dataset['classification'] = dataset['classification'].map({'yes': 1, 'no': 0})

# Separate features and target
independant = dataset.drop("classification", axis=1)
dependant = dataset["classification"]

# Identify feature types
numeric_features = independant.select_dtypes(include=["int64", "float64"]).columns
categorical_features = independant.select_dtypes(include=["object"]).columns

# Preprocessing Numerical and Categorical data
from sklearn.pipeline import Pipeline 
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="mean")),
    ("scaler", StandardScaler())
])

# Pre-Process Nominal data using one hot encoding
from sklearn.preprocessing import OneHotEncoder
categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

# Pre-Process Numerical and Categorical data
from sklearn.compose import ColumnTransformer
preprocessor = ColumnTransformer([
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])
#print(preprocessor)



In [2]:
# Executing the output data using pipeline and Naive Bayes


from sklearn.pipeline import Pipeline
from sklearn.naive_bayes import GaussianNB
pipeline = Pipeline([("preprocessor", preprocessor),("classifier", GaussianNB())])

# Grid Search parameters for Random Forest
param_grid = {"classifier__var_smoothing": [1e-9, 1e-8, 1e-7, 1e-6]}

# Train-test split
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(independant,dependant,test_size=0.33,random_state=42,
                                                    stratify=dependant)

# Grid Search - Creating and fitting the model
from sklearn.model_selection import GridSearchCV
grid_search = GridSearchCV(estimator=pipeline,param_grid=param_grid,cv=5,scoring="accuracy",n_jobs=-1)

grid_search.fit(X_train, y_train)

# Best model results
print("Best Parameters:")
print(grid_search.best_params_)

best_model = grid_search.best_estimator_
y_pred = best_model.predict(X_test)

Best Parameters:
{'classifier__var_smoothing': 1e-09}


In [3]:
##Printing Accuracy , Confusion Matric and Classification Report
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
accuracy=accuracy_score(y_test, y_pred)
print("\nAccuracy:", accuracy)

cm=confusion_matrix(y_test, y_pred)
print("\nConfusion Matrix:\n",cm)

creport=classification_report(y_test, y_pred)
print("\nClassification Report:\n", creport)


Accuracy: 0.9924242424242424

Confusion Matrix:
 [[50  0]
 [ 1 81]]

Classification Report:
               precision    recall  f1-score   support

           0       0.98      1.00      0.99        50
           1       1.00      0.99      0.99        82

    accuracy                           0.99       132
   macro avg       0.99      0.99      0.99       132
weighted avg       0.99      0.99      0.99       132

